In [1]:
DIRECTORY = "tmp/"


In [2]:
import os

logfiles = []

for folder in os.listdir(DIRECTORY):
    for file in os.listdir(os.path.join(DIRECTORY, folder)):
        if file.endswith(".log"):
            log_file_path = os.path.join(DIRECTORY, folder, file)
            logfiles.append(log_file_path)

logfiles

['tmp/4jcb0dbjyy/train.log',
 'tmp/wnwjbxmd0t/train.log',
 'tmp/u9aczfm6zj/train.log',
 'tmp/6gk8vnoyet/train.log',
 'tmp/hplu3s9t0d/train.log',
 'tmp/y4okbddeqo/train.log',
 'tmp/6r286eswe4/train.log',
 'tmp/jcmlb6nij9/train.log',
 'tmp/rub43ift2f/train.log',
 'tmp/vyobcoz7cl/train.log']

In [3]:
import re

def get_params(log_file_path):
    with open(log_file_path, 'r') as log_file:
        log_data = log_file.read()
    pattern = re.compile(r'(\w+):\s*(.+)')

    params = {}
    for match in pattern.finditer(log_data):
        key, value = match.groups()
        
        # If the value can be interpreted as a number (int or float), convert it
        if value.lower() in ['true', 'false']:
            value = value.lower() == 'true'
        elif re.match(r'^\d+(\.\d+)?$', value):  # If the value is a number (integer or float)
            value = float(value) if '.' in value else int(value)
        
        # Add to the dictionary
        params[key] = value
    return params


In [12]:
import re
import json

def get_training_testing_steps(log_file_path):
    training_steps = []
    testing_steps = []

    log_pattern = re.compile(r'__log:\{(.*?)\}')  # Matches everything between __log:{...}

    with open(log_file_path, 'r') as log_file:
        for line in log_file:
            match = log_pattern.search(line)
            if match:
                log_data = match.group(1) 
                try:
                    log_json = json.loads('{' + log_data + '}')
                    if 'train_acc' in log_json:  # This indicates a training step
                        training_steps.append(log_json)
                    elif 'test_acc_ema' in log_json:  # This indicates a testing step
                        testing_steps.append(log_json)
                except json.JSONDecodeError:
                    print(f"Error decoding JSON in line: {line.strip()}")

    return training_steps, testing_steps

log_file_path = 'tmp/6gk8vnoyet/train.log'
training_steps, testing_steps = get_training_testing_steps(log_file_path)
print(training_steps)

[{'nb_steps': 20, 'train_acc': 0.17161747699930427, 'loss': 2.2396337196302545, 'grad_sample_gradients_norms': 11.671040291619823, 'grad_sample_gradients_norms_lowerC': 0.0}, {'nb_steps': 40, 'train_acc': 0.24914960413094583, 'loss': 2.1085805913369433, 'grad_sample_gradients_norms': 10.865665763428831, 'grad_sample_gradients_norms_lowerC': 0.0}, {'nb_steps': 60, 'train_acc': 0.2582674895595204, 'loss': 2.10630708607797, 'grad_sample_gradients_norms': 11.297015499481978, 'grad_sample_gradients_norms_lowerC': 0.0005744925315970893}, {'nb_steps': 80, 'train_acc': 0.33291058995238887, 'loss': 1.8896698056881156, 'grad_sample_gradients_norms': 10.187074867046833, 'grad_sample_gradients_norms_lowerC': 0.0015794669299111549}, {'nb_steps': 100, 'train_acc': 0.36264116908783656, 'loss': 1.8224159106375553, 'grad_sample_gradients_norms': 11.197723247030538, 'grad_sample_gradients_norms_lowerC': 0.0027944111776447107}, {'nb_steps': 120, 'train_acc': 0.3863216232019323, 'loss': 1.7899618212447161

In [ ]:
log_file_path = 'tmp/6gk8vnoyet/train.log'
experiment1 = get_params(log_file_path)
training_steps, testing_steps = get_training_testing_steps(log_file_path)
experiment1["training_steps"] = training_steps
experiment1["testing_steps"] = testing_steps
experiment1

In [23]:
experiments = []
for log_file_path in logfiles:
    exp = get_params(log_file_path)
    training_steps, testing_steps = get_training_testing_steps(log_file_path)
    exp["training_steps"] = training_steps
    exp["testing_steps"] = testing_steps
    experiments.append(exp)

In [27]:
import pandas as pd
df_experiments = pd.DataFrame(experiments)
df_experiments.columns

Index(['00', 'WRN_k', 'batch_size', 'command', 'data_root', 'debug_slurm',
       'delta', 'dump_path', 'exp_id', 'exp_name', 'experiment', 'freq_log',
       'freq_log_val', 'global_rank', 'init', 'is_master', 'is_slurm_job',
       'local_rank', 'lr', 'master_port', 'max_per_sample_grad_norm',
       'max_physical_batch_size', 'momentum', 'multi_gpu', 'multi_node',
       'n_gpu_per_node', 'n_nodes', 'nb_groups', 'node_id', 'order1', 'order2',
       'poisson_sampling', 'proportion', 'ref_B', 'ref_nb_steps', 'ref_noise',
       'transform', 'world_size', '01', 'training_steps', 'testing_steps',
       '14', '15', '06', '07', '08', '09', '03', '04', '05', '02', '10', '11',
       '12', '13'],
      dtype='object')

In [28]:
df_experiments["ref_noise"]

0    3.0
1    1.0
2    3.0
3    3.0
4    3.0
5    1.0
6    3.0
7    3.0
8    1.0
9    3.0
Name: ref_noise, dtype: float64